In [1]:
#--------------------------------------------------------------------------------------------<Import libraries>--------------||
import numpy as np
import pandas as pd
import h5py
import os
import warnings
import torch
from torch.distributions import MultivariateNormal
from jakteristics import compute_features

#SUPPRESS WARNINGS
warnings.filterwarnings("ignore")

In [2]:
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU Name: NVIDIA GeForce RTX 4080 SUPER


In [3]:
fileName = '004'

global_asprs_classes = [0, 1, 2, 3, 4, 5, 6, 7, 8] 
num_global_classes = len(global_asprs_classes)

asprs_to_idx = {cls: i for i, cls in enumerate(global_asprs_classes)}

print('Done')

Done


In [4]:
Input_path = rf"F:\Aditya\Tiles\Toronto Tiles\features_L{fileName}.csv"

data = pd.read_csv(Input_path, sep=r',', engine='python')
data.columns = data.columns.str.replace('//', '', regex=False).str.strip().str.lower()
if 'classification' in data.columns:
    data = data.rename(columns={'classification': 'label'})
if any (col in data.columns for col in ['nx', 'ny', 'nz']):
    data = data.drop(columns=['nx', 'ny', 'nz'])
    
data.columns  

Index(['unnamed: 0', 'x', 'y', 'z', 'eigenvalue_sum', 'omnivariance',
       'eigenentropy', 'anisotropy', 'planarity', 'linearity', 'pca1', 'pca2',
       'surface_variation', 'sphericity', 'verticality', 'label'],
      dtype='object')

In [5]:
data['x'] -= data['x'].min()
data['y'] -= data['y'].min()
print('Done')

Done


In [6]:
data = data.astype(float)
grouped = data.groupby(data['label'])

#--------------------------------------------------------------------------------------------<Check distribution>------------||
averages = grouped.mean()
variances = grouped.var()
averages

,unnamed: 0,x,y,z,eigenvalue_sum,omnivariance,eigenentropy,anisotropy,planarity,linearity,pca1,pca2,surface_variation,sphericity,verticality
label,,,,,,,,,,,,,,,
0.0,3.537268e+06,131.300928,162.851660,136.167199,0.029127,0.004576,0.125160,0.921614,0.614821,0.306792,0.573610,0.384501,0.041889,0.078386,0.302150
1.0,3.531430e+06,129.741485,189.915674,135.262950,0.030859,0.002124,0.128763,0.994286,0.827440,0.166846,0.547643,0.449207,0.003151,0.005714,0.007181
2.0,3.537718e+06,131.009482,190.186262,135.280352,0.031035,0.002292,0.129474,0.996232,0.866019,0.130213,0.535430,0.462573,0.001997,0.003768,0.000583
3.0,3.038697e+06,138.747764,178.384079,141.541583,0.024500,0.005750,0.110015,0.821784,0.304359,0.517425,0.629000,0.273618,0.097381,0.178216,0.500229
4.0,3.707857e+06,126.982232,155.821370,140.670164,0.027797,0.003957,0.118793,0.954610,0.547636,0.406975,0.626681,0.346991,0.026328,0.045390,0.815768
5.0,3.097492e+06,127.230518,189.785511,143.136830,0.021341,0.002096,0.089836,0.976016,0.188657,0.787359,0.839579,0.144421,0.015999,0.023984,0.571248
6.0,2.680821e+06,126.847456,200.190288,139.982963,0.023840,0.004695,0.106410,0.921087,0.354117,0.566970,0.674300,0.275874,0.049827,0.078913,0.902396
7.0,1.459319e+06,116.014335,259.695496,135.824734,0.028215,0.004754,0.121776,0.929499,0.586901,0.342599,0.592943,0.368558,0.038498,0.070501,0.507315
8.0,2.554281e+06,94.690764,220.683280,136.893998,0.026719,0.003093,0.112796,0.970186,0.441844,0.528342,0.690212,0.291491,0.018297,0.029814,0.800026


In [7]:
def compute_covariance_matrix(data, regularization=1e-5):
    if data.shape[0] < 2:
        return np.eye(data.shape[1]) * regularization
    
    cov_matrix = np.cov(data, rowvar=False)
    cov_matrix += regularization * np.eye(cov_matrix.shape[0])
    
    return cov_matrix

print('Done')

Done


In [8]:
def fit(x_train, y_train):
    y_train = y_train.ravel()
    m = y_train.shape[0] 
    input_feature = x_train.shape[1]
    
    unique_labels = np.unique(y_train)
    num_classes = len(unique_labels)
    
    mu = np.zeros((num_classes, input_feature))
    sigma = np.zeros((num_classes, input_feature, input_feature))
    phi = np.zeros(num_classes)

    for idx, label_val in enumerate(unique_labels):
        indices = (y_train == label_val)
        x_class = x_train[indices, :]
        
        if len(x_class) > 1:
            phi[idx] = float(np.sum(indices)) / m
            mu[idx] = np.mean(x_class, axis=0)
            sigma[idx] = compute_covariance_matrix(x_class)
        else:
            phi[idx] = float(np.sum(indices)) / m
            mu[idx] = x_class[0] if len(x_class) == 1 else np.zeros(input_feature)
            sigma[idx] = np.eye(input_feature) * 1e-5
    
    return phi, mu, sigma, unique_labels

print('Done')

Done


In [9]:
has_nan = data.isnull().values.any()
print(has_nan)

data.drop('unnamed: 0', axis=1, inplace=True)
data['label'] = data['label'].astype(int)
data.columns

True


Index(['x', 'y', 'z', 'eigenvalue_sum', 'omnivariance', 'eigenentropy',
       'anisotropy', 'planarity', 'linearity', 'pca1', 'pca2',
       'surface_variation', 'sphericity', 'verticality', 'label'],
      dtype='object')

In [10]:
#--------------------------------------------------------------------------------------------<Nomralization>-----------------||
from sklearn.preprocessing import MinMaxScaler #scikit-learn
scaler = MinMaxScaler()

columns_to_scale = ['eigenvalue_sum', 'omnivariance', 'eigenentropy',
       'anisotropy', 'planarity', 'linearity', 'pca1', 'pca2',
       'surface_variation', 'sphericity', 'verticality'] #Toronto, excluded z

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data[columns_to_scale])
scaled_df = pd.DataFrame(scaled_data, columns=columns_to_scale)
data[columns_to_scale] = scaled_df
data

,x,y,z,eigenvalue_sum,omnivariance,eigenentropy,anisotropy,planarity,linearity,pca1,pca2,surface_variation,sphericity,verticality,label
0,61.237976,284.179016,137.710007,0.251382,0.156383,0.364325,0.961840,0.581106,0.383914,0.396741,0.746365,0.066442,0.038160,0.803205,6
1,61.361023,284.220032,136.304001,0.407563,0.205393,0.529990,0.979866,0.682714,0.299093,0.360065,0.815492,0.033668,0.020134,0.949523,6
2,61.304993,284.348999,136.171005,0.362773,0.226307,0.481564,0.961248,0.378816,0.585514,0.525098,0.571737,0.076826,0.038752,0.951077,6
3,61.289001,284.049011,137.479996,0.236926,0.135310,0.346216,0.970987,0.579202,0.394299,0.407441,0.742263,0.051100,0.029013,0.983630,6
4,61.270996,284.201050,137.345993,0.428145,0.156723,0.527056,0.991681,0.253689,0.738774,0.675889,0.411865,0.018852,0.008319,0.950888,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6747643,128.924988,123.722046,137.091003,0.331571,0.304500,0.465192,0.853159,0.376822,0.487275,0.398189,0.622119,0.256073,0.146841,0.949365,3
6747644,128.907043,123.777039,137.039993,0.421126,0.250438,0.544168,0.967240,0.530296,0.439696,0.432128,0.704727,0.059220,0.032760,0.858457,3
6747645,128.869995,123.834045,136.988007,0.551537,0.302091,0.662185,0.974697,0.433636,0.543204,0.503845,0.617398,0.049151,0.025303,0.862066,3
6747646,128.817993,123.947021,136.884995,0.419748,0.268063,0.545955,0.958428,0.583093,0.378765,0.392103,0.748833,0.072020,0.041572,0.772900,3


In [11]:
print(data['label'].value_counts())

label
1    3721434
3    1292437
0     592162
4     539074
2     274204
7     214958
6      71609
5      37201
8       4569
Name: count, dtype: int64


In [12]:
data = data.dropna(subset=columns_to_scale + ['label'])
X = pd.concat([data[['x', 'y', 'z']], data[columns_to_scale]], axis=1) #Toronto
y = data[['label']]

print('Done')

Done


In [13]:
#--------------------------------------------------------------------------------------------<Data split>--------------------||
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2) #REVERTTHIS

X_train_values = X_train.values
y_train_values = y_train.values

print('Done')

Done


In [14]:
%%time
phi, mu, sigma, label_order = fit(X_train_values, y_train_values)
phi

CPU times: total: 10.9 s
Wall time: 860 ms


array([0.08820953, 0.55527758, 0.04087204, 0.18704156, 0.07976232,
       0.00552291, 0.01071452, 0.03192725, 0.00067228])

In [15]:
#--------------------------------------------------------------------------------------------<Multivariate gaussian pdf>-----||
import math
def multivariate_gaussian_pdf(x, mean, cov):
    d = mean.shape[0] #dimensionality of the input
    exponent = -0.5 * np.dot(np.dot((x - mean).T, np.linalg.inv(cov)), (x - mean)) # -(1/2) . Transpose(x−μ) . Inverse(Σ or Covariance Matx) . (x−μ)
    prefactor = 1 / np.sqrt(((2 * np.pi) ** d )*(np.linalg.det(cov))) # 1 / Sqrt( (2π)^d . |Σ| ) 
    return np.exp(exponent)*prefactor

print('Done')

Done


In [16]:
#--------------------------------------------------------------------------------------------<Positive semi definite chck>---||
# Covariance matrices must be positive semidefinite because variance (and covariance) can't be negative
def is_positive_semidefinite(matrix):
    eigenvalues, _ = np.linalg.eig(matrix)
    print(eigenvalues)
    return np.all(eigenvalues >= 0)

matrix = sigma[1] 
# print(matrix)
positive_semidefinite = is_positive_semidefinite(matrix)
if positive_semidefinite:
    print("The matrix is positive semidefinite.")
else:
    print("The matrix is not positive semidefinite.")

[5.67179565e+03 4.42603397e+01 5.80160182e-02 2.81583693e-02
 5.82324216e-03 2.24967578e-03 1.56537502e-03 5.46235524e-04
 3.46294827e-04 1.60631674e-05 1.20851126e-05 1.00000000e-05
 1.00000000e-05 1.00000000e-05]
The matrix is positive semidefinite.


In [17]:
print(sigma)

[[[ 2.57934767e+02 -7.29161596e+02 -1.16958610e+00 ... -2.79713612e-01
   -1.87708193e-01 -1.73193635e+00]
  [-7.29161596e+02  3.84516744e+03 -9.21366261e+00 ...  5.01879802e-01
    3.52341660e-01  4.89411674e+00]
  [-1.16958610e+00 -9.21366261e+00  6.84975674e-01 ...  5.65929164e-02
    3.76694844e-02  1.12223542e-01]
  ...
  [-2.79713612e-01  5.01879802e-01  5.65929164e-02 ...  2.85668503e-02
    1.97044891e-02  2.47332068e-02]
  [-1.87708193e-01  3.52341660e-01  3.76694844e-02 ...  1.97044891e-02
    1.39311326e-02  1.64744828e-02]
  [-1.73193635e+00  4.89411674e+00  1.12223542e-01 ...  2.47332068e-02
    1.64744828e-02  1.25511638e-01]]

 [[ 3.51914389e+02 -1.27932502e+03  4.06075031e+00 ... -1.09029325e-02
   -8.57826521e-03 -2.02403036e-02]
  [-1.27932502e+03  5.36409142e+03 -1.62029911e+01 ...  2.80962525e-02
    2.37719690e-02  2.03085055e-03]
  [ 4.06075031e+00 -1.62029911e+01  7.83450512e-02 ...  2.30743525e-04
    1.30751705e-04  1.23717870e-03]
  ...
  [-1.09029325e-02  2.8

In [18]:
for i in range(sigma.shape[0]): 
    print(np.linalg.eigvals(sigma[i]))

[3.98774366e+03 1.15466708e+02 6.41722506e-01 1.10717041e-01
 1.04885090e-01 4.98518794e-02 8.73137697e-03 2.88516233e-03
 8.84302570e-04 2.09849309e-04 1.80188615e-05 1.00000000e-05
 1.00000000e-05 1.00000000e-05]
[5.67179565e+03 4.42603397e+01 5.80160182e-02 2.81583693e-02
 5.82324216e-03 2.24967578e-03 1.56537502e-03 5.46235524e-04
 3.46294827e-04 1.60631674e-05 1.20851126e-05 1.00000000e-05
 1.00000000e-05 1.00000000e-05]
[6.69653655e+03 2.41431923e+01 2.60904562e-02 7.83913987e-03
 1.19473295e-03 1.07015641e-03 1.56110662e-04 1.29481572e-04
 5.24670005e-05 1.03369873e-05 1.04746205e-05 1.00000000e-05
 1.00000000e-05 1.00000000e-05]
[4.66087107e+03 4.77669195e+02 1.60747034e+01 2.07386075e-01
 8.81439646e-02 7.34344119e-02 3.16791235e-02 3.41175014e-03
 1.76177045e-03 3.19684525e-04 4.82708962e-05 9.99999997e-06
 1.00000000e-05 1.00000000e-05]
[6.10655199e+03 4.60747243e+02 8.21468311e+00 1.99143019e-01
 6.34612992e-02 3.63856058e-02 1.34797049e-02 2.30425545e-03
 1.48174465e-03 1.

#### <span style="color:red">Epistemic uncertainty calculation</span>
Computes the epistemic uncertainty using this formula 1 − ∑ P( y=c | x )⋅ϕ(c)
- P( y=c | x ) is class probability from the Gaussian
- ϕ(c) is the prior (Φ, Prior probability of each class or Class prior probabilities (class frequencies))

In [19]:
def give_epistemic_torch(X_df, mu, sigma, phi, device='cuda'):
    x_test = torch.tensor(X_df.values, dtype=torch.float32).to(device)
    mu_t = torch.tensor(mu, dtype=torch.float32).to(device)
    sigma_t = torch.tensor(sigma, dtype=torch.float32).to(device)
    phi_t = torch.tensor(phi, dtype=torch.float32).to(device)

    num_pts = x_test.shape[0]
    num_classes = sigma_t.shape[0]

    total_weighted_density = torch.zeros(num_pts, device=device)
    sum_of_densities = torch.zeros(num_pts, device=device)

    print(f"Calculating uncertainty for {num_pts} points on {device}...")

    for idx in range(num_classes):
        if phi_t[idx] > 0:
            dist = MultivariateNormal(loc=mu_t[idx], covariance_matrix=sigma_t[idx])
            p_x_given_c = torch.exp(dist.log_prob(x_test))
            
            weighted_p = p_x_given_c * phi_t[idx]
            sum_of_densities += weighted_p
            total_weighted_density += weighted_p * phi_t[idx]

    sum_of_densities = torch.clamp(sum_of_densities, min=1e-10)
    
    feature_density = total_weighted_density / sum_of_densities
    uncertainty = 1 - feature_density

    return uncertainty.cpu().numpy()

print('Done')

Done


In [ ]:
%%time

#Compute epistemic
X_epistemic = give_epistemic_torch(X, mu, sigma, phi)

Calculating uncertainty for 6697335 points on cuda...


In [ ]:
X['epistemic'] = X_epistemic
data_new = pd.concat([X, y], axis=1)
data_new = data_new.dropna()
data_new.drop(columns=['eigenvalue_sum', 'pca1', 'pca2'], inplace=True)

print('Done')

In [ ]:
print(f"{fileName}: {data['label'].value_counts(normalize=True)}")

In [ ]:
data_new.to_csv(rf"F:\Aditya\Tiles\Toronto Tiles\features_L{fileName}.csv", index=False)
data_new.columns

print('Done')